# 산업규모별 임금 및 근로시간 데이터 분석

이 노트북은 2026년 1월 `산업규모별임금및근로시간.csv` 공공데이터를 사용해 산업별·사업체 규모별 임금과 근로시간 차이를 비교한다.

발표에서는 단순히 그래프를 설명하기보다 다음 질문에 답하는 흐름으로 진행한다.

1. 전체 사업체의 평균적인 근로시간과 임금 수준은 어느 정도인가?
2. 어떤 산업에서 임금이 높거나 낮게 나타나는가?
3. 사업체 규모가 커질수록 임금과 근로시간은 어떻게 달라지는가?
4. 임금 격차는 근로시간 차이 때문인지, 특별급여 등 임금 구성 차이 때문인지 확인한다.


In [9]:
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import platform

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 실행 환경에 따라 한글 폰트를 다르게 설정한다.
# Windows에서는 Malgun Gothic, macOS에서는 AppleGothic을 우선 사용한다.
system_name = platform.system()
if system_name == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif system_name == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')

plt.rc('axes', unicode_minus=False)
sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'][0])

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.1f}'.format)


## 1. 데이터 불러오기

CSV 파일은 `cp949` 인코딩이며, 첫 번째 행에 실제 지표명이 들어 있다. 노트북 위치가 바뀌어도 실행되도록 현재 폴더의 CSV 파일을 자동으로 찾는다.


In [10]:
raw = pd.read_csv('산업규모별임금및근로시간.csv', encoding='cp949')
raw.head()


FileNotFoundError: [Errno 2] No such file or directory: '산업규모별임금및근로시간.csv'

## 2. 데이터 정리

원본은 첫 행이 컬럼명처럼 들어가 있으므로 첫 행을 컬럼명으로 올리고, 공백이 섞인 규모명을 정리한 뒤 수치형 지표를 숫자로 변환한다.


In [ ]:
# 첫 번째 데이터 행을 실제 컬럼명으로 사용한다.
df = raw.iloc[1:].copy()
df.columns = raw.iloc[0].tolist()
df.columns.name = None
df = df.reset_index(drop=True)

# 규모별 값 앞뒤에 붙은 공백과 하이픈 표기 차이를 정리한다.
df['규모별'] = df['규모별'].str.strip().str.lstrip('-')

# 산업분류와 규모별을 제외한 지표 컬럼은 모두 숫자형으로 변환한다.
numeric_cols = df.columns[2:]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

df.head()


KeyError: '규모별'

In [ ]:
# 발표에서는 df.info() 전체 출력보다 핵심 점검값만 보여주는 편이 보기 쉽다.
data_check = pd.DataFrame({
    '항목': ['행 수', '열 수', '산업분류 수', '규모 구분 수', '결측치 수'],
    '값': [
        df.shape[0],
        df.shape[1],
        df['산업분류'].nunique(),
        df['규모별'].nunique(),
        int(df.isna().sum().sum()),
    ]
})

data_check


In [ ]:
# 누적 규모 구간과 실제 규모 구간이 함께 있으므로, 분석 목적에 맞게 구분해서 사용한다.
display(df['산업분류'].value_counts().to_frame('행 수').head())
display(pd.DataFrame({'규모별 구분': df['규모별'].unique()}))


In [ ]:
missing = df.isna().sum().to_frame('결측치 수')
missing[missing['결측치 수'] > 0]


## 3. 전체 현황

`산업분류 = 전체`, `규모별 = 전규모(1인이상)` 행을 기준으로 전체 사업체의 대표값을 확인한다.

In [ ]:
# 전체 산업, 전규모 행을 기준선으로 사용한다.
overall = df[(df['산업분류'] == '전체') & (df['규모별'] == '전규모(1인이상)')]
overall.T


In [ ]:
key_cols = [
    '전체근로시간 (시간)',
    '상용초과근로시간 (시간)',
    '전체임금총액 (원)',
    '상용정액급여 (원)',
    '상용초과급여 (원)',
    '상용특별급여 (원)',
]

# 주요 지표의 대략적인 분포를 확인한다.
df[key_cols].describe()


## 4. 산업별 비교

산업별 비교에서는 규모 조건이 섞이지 않도록 각 산업의 `전규모(1인이상)` 행만 사용한다.

In [ ]:
# 산업 간 비교에서는 규모가 섞이지 않도록 전규모(1인이상) 행만 사용한다.
industry_df = df[(df['산업분류'] != '전체') & (df['규모별'] == '전규모(1인이상)')].copy()

# 단순 시간당임금은 임금총액을 전체근로시간으로 나눈 참고용 지표다.
# 실제 시급과는 계산 기준이 다를 수 있으므로 발표에서는 '단순 추정'이라고 설명한다.
industry_df['시간당임금_단순추정'] = industry_df['전체임금총액 (원)'] / industry_df['전체근로시간 (시간)']
industry_df['특별급여비중(%)'] = industry_df['상용특별급여 (원)'] / industry_df['상용임금총액 (원)'] * 100
industry_df['초과근로비중(%)'] = industry_df['상용초과근로시간 (시간)'] / industry_df['상용총근로시간 (시간)'] * 100

industry_summary = industry_df[[
    '산업분류',
    '전체근로시간 (시간)',
    '상용초과근로시간 (시간)',
    '초과근로비중(%)',
    '전체임금총액 (원)',
    '상용정액급여 (원)',
    '상용초과급여 (원)',
    '상용특별급여 (원)',
    '특별급여비중(%)',
    '시간당임금_단순추정',
]]

industry_summary.sort_values('전체임금총액 (원)', ascending=False).head(10)


In [ ]:
print('전체임금총액 하위 5개 산업')
display(industry_summary.sort_values('전체임금총액 (원)').head(5))

print('전체근로시간 상위 5개 산업')
display(industry_summary.sort_values('전체근로시간 (시간)', ascending=False).head(5))

print('단순 시간당임금 상위 5개 산업')
display(industry_summary.sort_values('시간당임금_단순추정', ascending=False).head(5))


In [ ]:
top10_wage = industry_summary.sort_values('전체임금총액 (원)', ascending=False).head(10)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=top10_wage.sort_values('전체임금총액 (원)'),
    x='전체임금총액 (원)',
    y='산업분류',
    color='steelblue'
)
plt.title('산업별 전체임금총액 상위 10개 산업')
plt.xlabel('전체임금총액 (원)')
plt.ylabel('')
plt.ticklabel_format(style='plain', axis='x')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=industry_summary,
    x='전체근로시간 (시간)',
    y='전체임금총액 (원)',
    size='상용초과근로시간 (시간)',
    hue='상용특별급여 (원)',
    palette='viridis',
    sizes=(60, 260)
)
plt.title('산업별 근로시간과 임금총액 관계')
plt.xlabel('전체근로시간 (시간)')
plt.ylabel('전체임금총액 (원)')
plt.ticklabel_format(style='plain', axis='y')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 5. 규모별 비교

누적 구간이 아닌 실제 규모 구간(`1~4인`, `5~9인`, `10~29인`, `30~99인`, `100~299인`, `300인이상`)만 비교한다.

In [ ]:
size_order = ['1~4인', '5~9인', '10~29인', '30~99인', '100~299인', '300인이상']

# 규모별 비교는 누적 구간이 아닌 실제 규모 구간만 남긴다.
size_df = df[(df['산업분류'] == '전체') & (df['규모별'].isin(size_order))].copy()
size_df['규모별'] = pd.Categorical(size_df['규모별'], categories=size_order, ordered=True)
size_df = size_df.sort_values('규모별')

base_wage = size_df.loc[size_df['규모별'] == '1~4인', '전체임금총액 (원)'].iloc[0]
size_df['임금_1~4인대비'] = size_df['전체임금총액 (원)'] / base_wage
size_df['특별급여비중(%)'] = size_df['상용특별급여 (원)'] / size_df['상용임금총액 (원)'] * 100

size_summary = size_df[[
    '규모별',
    '전체근로시간 (시간)',
    '상용초과근로시간 (시간)',
    '전체임금총액 (원)',
    '임금_1~4인대비',
    '상용정액급여 (원)',
    '상용초과급여 (원)',
    '상용특별급여 (원)',
    '특별급여비중(%)',
]]
size_summary


In [ ]:
size_labels = size_df['규모별'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(size_labels, size_df['전체임금총액 (원)'], marker='o', color='darkgreen')
axes[0].set_title('규모별 전체임금총액')
axes[0].set_xlabel('규모별')
axes[0].set_ylabel('전체임금총액 (원)')
axes[0].ticklabel_format(style='plain', axis='y')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(size_labels, size_df['전체근로시간 (시간)'], color='darkorange', alpha=.8)
axes[1].set_title('규모별 전체근로시간')
axes[1].set_xlabel('규모별')
axes[1].set_ylabel('전체근로시간 (시간)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


### 규모별 격차 해석

300인이상 사업체는 1~4인 사업체보다 임금총액이 약 3배 높다. 근로시간 차이는 15.4시간 정도이므로, 발표에서는 단순히 오래 일해서 임금이 높은 것이 아니라 특별급여와 임금 구성 차이가 격차를 키운다는 점을 강조한다.


In [ ]:
gap_summary = size_summary[[
    '규모별',
    '전체근로시간 (시간)',
    '전체임금총액 (원)',
    '임금_1~4인대비',
    '상용특별급여 (원)',
    '특별급여비중(%)',
]].copy()

gap_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(
    data=size_df,
    x='규모별',
    y='임금_1~4인대비',
    order=size_order,
    color='seagreen',
    ax=axes[0]
)
axes[0].axhline(1, color='gray', linestyle='--', linewidth=1)
axes[0].set_title('1~4인 대비 임금 배율')
axes[0].set_xlabel('규모별')
axes[0].set_ylabel('배율')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(
    data=size_df,
    x='규모별',
    y='특별급여비중(%)',
    order=size_order,
    color='slateblue',
    ax=axes[1]
)
axes[1].set_title('규모별 특별급여 비중')
axes[1].set_xlabel('규모별')
axes[1].set_ylabel('상용임금총액 대비 비중(%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
exact_industry_size_df = df[(df['산업분류'] != '전체') & (df['규모별'].isin(size_order))].copy()
exact_industry_size_df['규모별'] = pd.Categorical(
    exact_industry_size_df['규모별'],
    categories=size_order,
    ordered=True
)

plt.figure(figsize=(9, 4))
sns.boxplot(
    data=exact_industry_size_df,
    x='규모별',
    y='전체임금총액 (원)',
    order=size_order,
    color='lightblue'
)
plt.title('산업별 분포로 본 규모별 전체임금총액')
plt.xlabel('규모별')
plt.ylabel('전체임금총액 (원)')
plt.ticklabel_format(style='plain', axis='y')
plt.tight_layout()
plt.show()


## 6. 주요 지표 상관관계

상관계수와 히트맵으로 임금·근로시간 지표의 관계를 확인한다. 이 부분은 결론을 보조하는 자료이므로 발표 시간이 부족하면 간단히 넘어가도 된다.


In [ ]:
corr = df[key_cols].corr()
corr


In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=.5,
    fmt='.2f'
)
plt.title('임금 및 근로시간 주요 지표 상관관계')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 분석 요약

- 2026년 1월 전체 산업 전규모 기준 전체근로시간은 158.0시간, 전체임금총액은 4,588,273원이다.
- 산업별 전규모 기준 전체임금총액은 `K.금융 및 보험업`이 약 936만원으로 가장 높고, `I.숙박 및 음식점업`이 약 241만원으로 가장 낮다.
- 전체근로시간은 `E.수도, 하수 및 폐기물 처리, 원료 재생업`, `C.제조업`, `B.광업` 순으로 길지만, 근로시간이 길다고 항상 임금총액이 가장 높은 것은 아니다.
- 전체 산업 기준 300인이상 사업체의 전체임금총액은 약 836만원으로 1~4인 사업체 약 278만원의 약 3.0배이다.
- 300인이상 구간은 상용특별급여가 약 325만원으로 매우 높아 임금총액 격차를 키운다. 따라서 규모별 임금 격차는 근로시간 차이보다 임금 구성 차이의 영향이 더 커 보인다.
- 전체근로시간과 전체임금총액의 상관은 중간 이하이며, 전체임금총액은 상용정액급여와 상용특별급여와의 관계가 더 강하다.

## 발표 흐름 추천

1. 데이터 소개: 2026년 1월 산업별·규모별 임금 및 근로시간 공공데이터라고 설명한다.
2. 전처리 설명: 첫 행을 컬럼명으로 바꾸고, 규모별 공백·하이픈을 정리했으며, 지표 컬럼을 숫자형으로 변환했다고 말한다.
3. 전체 현황: 전체 산업 기준 근로시간 158.0시간, 임금총액 약 459만원을 기준선으로 제시한다.
4. 산업별 비교: 금융·보험업이 가장 높고 숙박·음식점업이 가장 낮다는 점을 막대그래프로 설명한다.
5. 규모별 비교: 300인이상 사업체의 임금총액이 1~4인 사업체의 약 3배라는 점을 강조한다.
6. 해석: 근로시간 차이만으로는 임금 격차를 설명하기 어렵고, 특별급여 등 임금 구성 차이가 중요하다고 마무리한다.
